# Exploring the stochastic world of SGD - Neural Newtork Experiment

In this notebook, we empirically investigate the stochastic properties of **Stochastic Gradient Descent (SGD)** discussed in the previous sections by training a simple neural network on standard benchmark datasets.

The objective is to bridge the gap between the theoretical SDE interpretation of SGD and its practical behavior in machine learning applications. In particular, we analyze how stochasticity affects the optimization dynamics, convergence behavior, and stability of neural network training.

Through a series of controlled experiments, we compare deterministic and stochastic optimization methods and study the extent to which the theoretical stochastic properties emerge in realistic training scenarios.

This notebook is divided into three main parts:

1. **Neural Network Architecture**

   Implementation and discussion of the neural network model used in the experiments.

2. **Benchmark Datasets**

   Description of the datasets and experimental setup.

3. **Empirical Validation of Stochastic Properties**

   Experimental analysis and discussion of the stochastic behavior predicted by the SDE framework.

# Neural Network

We will use a really simple MLP and a simple CNN. Our main goal is to see the effect of the batch size / learning rate in order to prove if what we have seen up until know is still valid for neural network.

## MLP

We begin by setting up a clean PyTorch implementation of a simple Multi-Layer Perceptron (MLP) with explicit logging parameters. This allows us to intercept and compute the exact individual per-sample gradients needed to analyze the true covariance matrix $\Sigma(\theta)$ at any point during training.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import copy
import time

# Ensure strict deterministic behavior for empirical validation consistency
torch.manual_seed(37)
np.random.seed(37)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(37)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {device}")

Running on device: cuda


One important aspect for our empirical validation is the ability to extract **per-sample gradients** $\nabla \ell_i(\theta)$ to measure the true gradient noise covariance matrix:

$$
\Sigma(\theta) = \frac{1}{N}\sum_{i=1}^N \left(\nabla \ell_i(\theta) - \nabla L(\theta)\right)\left(\nabla \ell_i(\theta) - \nabla L(\theta)\right)^T
$$

We implement this using PyTorch's native `torch.func` module to perform vectorised jacobian mappings without polluting our standard forward pass or manually loop-cloning networks.

In [13]:
class EmpiricalMLP(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=4, output_dim=10):
        super(EmpiricalMLP, self).__init__()
        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        x = self.flatten(x)
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)
        return x

def compute_per_sample_gradients(model, inputs, targets, criterion):
    """
    Computes exact per-sample gradients for a batch using PyTorch functional vmap.
    Returns a 2D tensor of shape (batch_size, num_parameters).
    """
    # Isolate functional parameters and network state
    params = tuple(model.parameters())
    
    # Define a stateless single-sample loss function
    def compute_single_loss(params_flat, single_input, single_target):
        # Reconstruct stateful execution for functional tracking
        functional_model = copy.deepcopy(model)
        # Re-link parameters
        with torch.no_grad():
            for p, p_new in zip(functional_model.parameters(), params_flat):
                p.copy_(p_new)
        out = functional_model(single_input.unsqueeze(0))
        return criterion(out, single_target.unsqueeze(0))

    # Vectorize the gradient extraction over the batch dimension
    ft_compute_grad = torch.func.vmap(torch.func.grad(compute_single_loss), in_dims=(None, 0, 0))
    per_sample_grads_tuple = ft_compute_grad(params, inputs, targets)
    
    # Flatten and concatenate parameters into unified vectors per sample
    flattened_grads = []
    batch_size = inputs.size(0)
    for g in per_sample_grads_tuple:
        flattened_grads.append(g.view(batch_size, -1))
        
    return torch.cat(flattened_grads, dim=1)

# Benchmark

We will be using the **MNIST** dataset. For the high-dimensional covariance experiment, we can subsample or compress structural dimensionality to keep the exact matrix operations computationally trackable while retaining realistic data-driven cross-correlations.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
# A more optimized modern approach avoiding deepcopy
from torch.func import functional_call, vmap, grad

# Extract a static subset for computing full dataset true gradients $\nabla L(\theta)$
subset_size = 2000
full_eval_loader = DataLoader(TensorDataset(*[train_dataset.data[:subset_size].float().unsqueeze(1)/255.0 - 0.1307, 
                                              train_dataset.targets[:subset_size]]), 
                              batch_size=subset_size, shuffle=False)

# Empirical Validation

In this section we will try to empirically validate some of the results seen up until now.

## Macro-Time Scaling Rule

In the first notebook we found that when transforming discrete SGD iterations $k$ into a continuous SDE trajectory $x_t$, we equate a single update step to a tiny continuous physical time increment:
$$
\Delta t = \eta
$$
This implies that after $k$ total discrete iterations, the accumulated continuous macro-time elapsed is $t = k \cdot \eta$. Furthermore, when accounting for mini-batch sizes $S$ sampled from a dataset of size $N$, the physical diffusion strength scales directly with the ratio $\frac{\eta}{S}$.

We are gonna try if two networks trained with completely different discrete hyperparameters ($\eta_1, S_1$ vs $\eta_2, S_2$) will follow the *same trajectory profile* if their paths are parameterized and evaluated against continuous macro-time $t$, while the ratio $\frac{\eta}{S}$ is preserved. 

In [ ]:
def train_macro_time_experiment(config, max_macro_time=5.0):
    model = EmpiricalMLP().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=config['lr'])
    
    loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
    
    history_loss = []
    history_time = []
    
    current_macro_time = 0.0
    discrete_step = 0
    
    # Calculate continuous time delta per batch step: dt = lr / batch_size
    dt = config['lr'] / config['batch_size']
    
    running = True
    while running:
        for inputs, targets in loader:
            if current_macro_time > max_macro_time:
                running = False
                break
                
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            current_macro_time += dt
            discrete_step += 1
            
            if discrete_step % 10 == 0:
                history_loss.append(loss.item())
                history_time.append(current_macro_time)
                
    return np.array(history_time), np.array(history_loss)

# Config A Small Learning Rate, Small Batch Size
config_A = {'lr': 0.01, 'batch_size': 32} # Ratio = 0.0003125
# Config B Large Learning Rate, Large Batch Size (Identical ratio)
config_B = {'lr': 0.04, 'batch_size': 128} # Ratio = 0.0003125

print("Training System A...")
times_A, losses_A = train_macro_time_experiment(config_A)
print("Training System B...")
times_B, losses_B = train_macro_time_experiment(config_B)

plt.figure(figsize=(10, 5))
plt.plot(times_A, losses_A, label=fr"System A ($\eta$={config_A['lr']}, S={config_A['batch_size']})", alpha=0.75)
plt.plot(times_B, losses_B, label=fr"System B ($\eta$={config_B['lr']}, S={config_B['batch_size']})", alpha=0.75)
plt.title(r"Empirical Validation of Macro-Time Scaling ($t = k \cdot \frac{\eta}{S}$)")
plt.xlabel(r"Continuous SDE Macro-Time ($t$)")
plt.ylabel(r"Loss")
plt.grid(True)
plt.legend()
plt.show()